# 06 · Conseguindo as imagens — **sem fotografar nada**

Rodar **em casa, uma vez.** Não é notebook de palco.

O problema: treinar um especialista exige centenas de imagens, e fotografar
isso à mão é inviável. A solução: o **Open Images**, um banco público do Google
com milhões de fotos **já anotadas**, do qual dá para puxar só a classe que
interessa.

Aqui você baixa garrafas de vinho, exporta no formato que o YOLO entende, e
ainda gera os recortes prontos para separar em *aberta* / *lacrada*.

> Fala de palco, se alguém perguntar de onde vêm os dados: *"não fotografei
> nada. Existe um banco público com milhões de imagens já marcadas, e eu puxei
> só o que precisava. O gargalo de um projeto de visão raramente é o modelo —
> é o dado. E dado bom, às vezes, já existe de graça."*

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

In [ ]:
# ── a ferramenta que fala com o Open Images ──
%pip install -q fiftyone
import fiftyone as fo
import fiftyone.zoo as foz
print("fiftyone", fo.__version__)

### Escolha o que baixar

O Open Images tem hierarquia de classes: **Bottle** é o guarda-chuva, e
**Wine** é a bebida (aparece em taça e em garrafa). Para contagem de estoque,
`Bottle` costuma render mais caixas úteis; `Wine` traz o contexto de adega e
mesa posta.

Comece com 600–1000 imagens. Mais que isso deixa o download longo e não muda
o resultado de uma demo.

In [ ]:
CLASSES = ["Bottle", "Wine"]      # ← mexa aqui
QUANTAS = 800                      # ← e aqui

dados = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=CLASSES,
    max_samples=QUANTAS,
    only_matching=True,            # só imagens que TÊM essas classes
    dataset_name="garrafas-palestra",
    overwrite=True,
)
print(dados)

In [ ]:
# ── exporta no formato que o YOLO entende, direto no Drive ──
import os
DESTINO = f"{DRIVE}/04-garrafas/dataset-yolo"
os.makedirs(DESTINO, exist_ok=True)

for split, fatia in (("train", 0.8), ("val", 0.2)):
    pass   # a divisão é feita abaixo, de uma vez

# embaralha e divide 80/20
import fiftyone.utils.random as four
four.random_split(dados, {"train": 0.8, "val": 0.2})

for split in ("train", "val"):
    parte = dados.match_tags(split)
    parte.export(
        export_dir=DESTINO,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        split=split,
        classes=CLASSES,
    )
print("exportado para", DESTINO)
print(os.listdir(DESTINO))

### Treinar o detector de garrafa (opcional)

O modelo pronto já detecta `bottle` muito bem — este treino serve para
**mostrar o processo**, não porque seja necessário. Se o seu objetivo é a demo
de estoque, pule: o `yolo11n.pt` resolve.

Treine se quiser um detector especializado em **garrafa de vinho** (que o COCO
não separa de garrafa de água).

In [ ]:
EPOCAS = 30

det = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
res = det.train(
    data=f"{DESTINO}/dataset.yaml",
    epochs=EPOCAS, imgsz=640, batch=16,
    project="/content/runs", name="garrafas-det", exist_ok=True,
    verbose=False, plots=True,
)
print("pesos em", res.save_dir)

### Os recortes para o classificador *aberta* / *lacrada*

Esta parte resolve o que faltava: o Open Images sabe que **é uma garrafa**, mas
não sabe se ela está **aberta**. Isso ninguém tem pronto — é específico do seu
problema, e por isso é o que vale.

A célula abaixo recorta cada garrafa encontrada e joga tudo numa pasta única.
Aí você separa em duas pastas, no próprio Drive, arrastando com o mouse. É
trabalhoso? É. **Mas é uma vez, e é aí que mora a sua vantagem.**

In [ ]:
# ── recorta cada garrafa e deixa pronto para você separar ──
import glob, os, cv2
from ultralytics import YOLO

detector = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
SAIDA = f"{DRIVE}/04-garrafas/recortes-para-separar"
os.makedirs(SAIDA, exist_ok=True)

fontes = []
for padrao in ("images/train/*", "images/val/*"):
    fontes += glob.glob(os.path.join(DESTINO, padrao))
fontes = [f for f in fontes if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"{len(fontes)} imagens de origem")

n = 0
for caminho in fontes[:400]:
    img = cv2.imread(caminho)
    if img is None:
        continue
    r = detector.predict(img, conf=.4, classes=[39], verbose=False)[0]
    for k, caixa in enumerate(r.boxes.xyxy.cpu().numpy().astype(int)):
        x1, y1, x2, y2 = caixa
        recorte = img[max(0, y1):y2, max(0, x1):x2]
        # descarta recorte minúsculo: não dá para ver a tampa, não serve
        if recorte.size == 0 or recorte.shape[0] < 64 or recorte.shape[1] < 24:
            continue
        cv2.imwrite(os.path.join(SAIDA, f"g{n:05d}.jpg"), recorte)
        n += 1

print(f"{n} recortes em {SAIDA}")
print("\nAgora, no Drive, separe em:")
print(f"  04-garrafas/treino/train/lacrada   e   .../aberta")
print(f"  04-garrafas/treino/val/lacrada     e   .../aberta   (uns 20% do total)")

---

## Se quiser mais imagens ainda

- **Open Images V7** — o que este notebook usa.
  <https://storage.googleapis.com/openimages/web/index.html>
  Explore as classes em `Bottle`, `Wine`, `Beer`, `Drink`.
- **Roboflow Universe** — <https://universe.roboflow.com>
  Busque por *bottle*, *wine bottle*, *retail shelf*, *supermarket*. Muitos
  datasets já vêm em formato YOLO, com download direto.
- **COCO** — a classe `bottle` já está dentro do modelo que você usa.

> **Cuidado com licença.** Open Images é CC-BY (dá para usar, citando a fonte).
> Datasets do Roboflow variam — a licença aparece na página de cada um. Para
> uma demo de palestra não muda nada, mas se virar produto de cliente, confira.